# Step 3: Make Some Objects for a Context Sighting and Save to Unattached Context Memory

## Key Insight

This system is designed to simulate the use of Python blocks, small scripts that will later be placed inside TotalJS Flow blocks. The invoke functions in the utilities collect the inputs and unwrap the outputs of the blocks, and the notebooks sequence the creation and saving of objects.

Context data is created in 3 stages, intially data is created in unattached context memory, then  promoted into the incident context memory. Incidents and company details can be Published from context memory into TypeDB for dissemination.

### Stage 1: Unattached Context -> Stored locally only

In the real system, objects are created through form data and stored as a list of objects in the unattached context file. As objects are connected together, and new objects created, eventually a subgraph will be formed based on one of the Promotable Objects:
- attack-flow
- behavior
- sighting
- task
- event
- impact

### Stage 2: Promoted to Incident Context -> Stored officially on the incident

Once one has formed a subgraph based on a Promotable object, then using the promotable block code enables the subgraph to be moved from unattached context memory, and saved into the incident replacing any existing copies of those objects. 


### Stage 3: Published to TypeDB

The context memory map Orchestration\generated\os-triage\context_mem\context_map.json, contains variables that are updateed every time an object is saved. Thus it is always clear which incidents, company's, team or user details need to be Published to TypeDB.


## Step 3. Notebook -> Get the Context to See Who Else Got the Email

Notebook to build all of the different stix objects associated with describing who else got the suspicious email. A quick loook through the Microsoft Ecyhange server showed that User 2, User 3, User 4 and User 5 all recieved the same suspicious email that User 1 got.

1. First setup the global parameters and retrieve the context memory
2. Retrieve the user-account, email address and identity for each of the 4 users who recieved the email
3. Add the SCO objects to the `ObservedData` object
4. Retrieve the Indicator object from the Alert created in Step 1
5. Create the `Sighting` object with the `SightingContext` extension, using their identity's as the location
6. Promote the Sighting from Unattached to the Incident Context Memory
7. Add the Sighting to the original Event
8. Create the next Task and its Sequence objects, and chain the Sequence object
9. Significantly expanded Impact object's with different extensions (e.g. Physical, monetary etc.), as there are now 5 lapptops impacted


Each time a block makes an object, and saves it as a json, this notebook will parse the object into an actual Stix object, so it can be bundled and printy printed (This step also verifies the objects are created correctly, and is cool).



## A. Load Imports

### A.1 StixORM Imports


In [1]:
import sys
!{sys.executable} -m pip install stixorm



[notice] A new release of pip is available: 24.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### A.2 Relative Path Imports
https://stackoverflow.com/questions/76162459/jupyter-notebook-importerror-attempted-relative-import-with-no-known-parent-pac/77528726#77528726

In [2]:
import sys
sys.path.append('../')
import os

### A.3 Relative Import of Blocks

In [3]:
import json
from Block_Families.StixORM.SCO.EmailMessage.make_email_message import main as make_email_msg
from Block_Families.StixORM.SRO.Relationship.make_sro import main as make_sro
from Utilities.local_make_general import invoke_save_incident_context_block, invoke_get_from_company_block, invoke_get_from_incident_block
from Utilities.local_make_general import invoke_move_unattached_to_other_block, invoke_chain_sequence_block, save_object_to_file, invoke_save_unattached_context_block
from Utilities.local_make_sro import invoke_sro_block, invoke_sighting_block
from Utilities.local_make_sdo import (
    invoke_make_observed_data_block, invoke_make_indicator_block, invoke_make_event_block, invoke_make_sequence_block,
    invoke_make_task_block, invoke_make_incident_block, invoke_make_impact_block
)
from Utilities.local_make_sco import (
    invoke_make_email_addr_block, invoke_make_url_block, invoke_make_e_msg_block, invoke_make_anecdote_block
)
from conv import conv

context_base = "../Orchestration/Context_Mem/"
path_base = "../Block_Families/Objects/"
results_base = "../Orchestration/Results/"
sro_data = "SRO/Relationship/sro_derived.json"
context_data = {
    "usr1": "naive@example.com",
    "usr2": "sbilly@example.com",
    "usr3": "wwhilly@example.com",
    "usr4": "strange@mycompany.com",
    "usr5": "dumbo@mycompany.com",
}

TR_Context_Memory_Path = "./Context_Mem/Type_Refinery_Context.json"

## B. Collect the SCO's Based on Searching the Exchange Server

### B.1 Load the Email Addresses

In [4]:
# Create the Queries to get the User Details

context_type = {
    "context_type": "users"
}
usr1_email_query = {
    "type" : "email-addr",
    "property": {
        "path": ["value"],
        "source_value": context_data["usr1"],
        "comparator": "EQ"
    }
}
usr1_email = invoke_get_from_company_block(usr1_email_query, context_type, source_value=None, source_id=None)
print(f"\n usr1_email->{usr1_email}")
email1_results_obj_path = results_base + "/step4/Usr1__email.json"
save_object_to_file(usr1_email, email1_results_obj_path)
usr2_email_query = {
    "type" : "email-addr",
    "property": {
        "path": ["value"],
        "source_value": context_data["usr2"],
        "comparator": "EQ"
    }
}
usr2_email = invoke_get_from_company_block(usr2_email_query, context_type, source_value=None, source_id=None)
print(f"\n usr2_email->{usr2_email}")
email2_results_obj_path = results_base + "/step4/Usr2__email.json"
save_object_to_file(usr2_email, email2_results_obj_path)
usr3_email_query = {
    "type" : "email-addr",
    "property": {
        "path": ["value"],
        "source_value": context_data["usr3"],
        "comparator": "EQ"
    }
}

usr3_email = invoke_get_from_company_block(usr3_email_query, context_type, source_value=None, source_id=None)
print(f"\n usr3_email->{usr3_email}")
email3_results_obj_path = results_base + "/step4/Usr3__email.json"
save_object_to_file(usr3_email, email3_results_obj_path)
usr4_email_query = {
    "type" : "email-addr",
    "property": {
        "path": ["value"],
        "source_value": context_data["usr4"],
        "comparator": "EQ"
    }
}
usr4_email = invoke_get_from_company_block(usr4_email_query, context_type, source_value=None, source_id=None)
print(f"\n usr4_email->{usr4_email}")
email4_results_obj_path = results_base + "/step4/Usr4__email.json"
save_object_to_file(usr4_email, email4_results_obj_path)
usr5_email_query = {
    "type" : "email-addr",
    "property": {
        "path": ["value"],
        "source_value": context_data["usr5"],
        "comparator": "EQ"
    }
}
usr5_email = invoke_get_from_company_block(usr5_email_query, context_type, source_value=None, source_id=None)
print(f"\n usr5_email->{usr5_email}")
email5_results_obj_path = results_base + "/step4/Usr5__email.json"
save_object_to_file(usr5_email, email5_results_obj_path)

#
# 3. save the objects to "unattached"
email_results_context_path = results_base + "/step4/context/usr1__email_context.json"
result1 = invoke_save_unattached_context_block(email1_results_obj_path, email_results_context_path)
print(f" result1->{result1}")
email_results_context_path = results_base + "/step4/context/usr2__email_context.json"
result2 = invoke_save_unattached_context_block(email2_results_obj_path, email_results_context_path)
print(f" result2->{result2}")
email_results_context_path = results_base + "/step4/context/usr3__email_context.json"
result3 = invoke_save_unattached_context_block(email3_results_obj_path, email_results_context_path)
print(f" result3->{result3}")
email_results_context_path = results_base + "/step4/context/usr4__email_context.json"
result4 = invoke_save_unattached_context_block(email4_results_obj_path, email_results_context_path)
print(f" result4->{result4}")
email_results_context_path = results_base + "/step4/context/usr5__email_context.json"
result5 = invoke_save_unattached_context_block(email5_results_obj_path, email_results_context_path)
print(f" result5->{result5}")
#
# Save everything but the existing indicator down itno unattached
# context_type = {
#     "context_type": "unattached"
# }
# sro2_results_path = results_base + "step4/observation-context_context.json"
# sro2_context_results_path = results_base + "step4/observation-context_context.json"
# result = invoke_save_incident_context_block(sro2_results_path, sro2_context_results_path, context_type)
# print(f"SRO2 result->{result}")

company query->{'type': 'email-addr', 'property': {'path': ['value'], 'source_value': 'naive@example.com', 'comparator': 'EQ'}}

 usr1_email->{'type': 'email-addr', 'spec_version': '2.1', 'id': 'email-addr--4722424c-7012-56b0-84d5-01d076fc547b', 'value': 'naive@example.com', 'display_name': 'Naive Smith', 'belongs_to_ref': 'user-account--597ad4d4-35ba-585d-8f6d-134a75032f9b'}
company query->{'type': 'email-addr', 'property': {'path': ['value'], 'source_value': 'sbilly@example.com', 'comparator': 'EQ'}}

 usr2_email->{'type': 'email-addr', 'spec_version': '2.1', 'id': 'email-addr--b0708db8-71e9-50f9-950c-610fccd2c30b', 'value': 'sbilly@example.com', 'display_name': 'Strange Billy', 'belongs_to_ref': 'user-account--5c0dcb9b-5784-5aaf-b393-d990a4d68dd3'}
company query->{'type': 'email-addr', 'property': {'path': ['value'], 'source_value': 'wwhilly@example.com', 'comparator': 'EQ'}}

 usr3_email->{'type': 'email-addr', 'spec_version': '2.1', 'id': 'email-addr--72fa6570-cce5-5be5-916d-452de



### B.2 Retrieve the User Account and Identity for each email address

for each user who got the phishing email

In [5]:
# 1. For user 2, get the user account
usr2_usr_acct_query = {
    "type" : "user-account",
    "property": {
        "path": ["id"],
        "source_value": usr2_email["belongs_to_ref"],
        "comparator": "EQ"
    }
}
usr2_acct = invoke_get_from_company_block(usr2_usr_acct_query, context_type, source_value=None, source_id=None)
usr2_acct_results_obj_path = results_base + "/step4/Usr2__acct.json"
save_object_to_file(usr2_acct, usr2_acct_results_obj_path)
# 2. For user 3, get the user account
usr3_usr_acct_query = {
    "type" : "user-account",
    "property": {
        "path": ["id"],
        "source_value": usr3_email["belongs_to_ref"],
        "comparator": "EQ"
    }
}
usr3_acct = invoke_get_from_company_block(usr3_usr_acct_query, context_type, source_value=None, source_id=None)
usr3_acct_results_obj_path = results_base + "/step4/Usr3__acct.json"
save_object_to_file(usr3_acct, usr3_acct_results_obj_path)
# 3. For user 4, get the user account
usr4_usr_acct_query = {
    "type" : "user-account",
    "property": {
        "path": ["id"],
        "source_value": usr4_email["belongs_to_ref"],
        "comparator": "EQ"
    }
}
usr4_acct = invoke_get_from_company_block(usr4_usr_acct_query, context_type, source_value=None, source_id=None)
usr4_acct_results_obj_path = results_base + "/step4/Usr4__acct.json"
save_object_to_file(usr4_acct, usr4_acct_results_obj_path)
# 4. For user 5, get the user account
usr5_usr_acct_query = {
    "type" : "user-account",
    "property": {
        "path": ["id"],
        "source_value": usr5_email["belongs_to_ref"],
        "comparator": "EQ"
    }
}
usr5_acct = invoke_get_from_company_block(usr5_usr_acct_query, context_type, source_value=None, source_id=None)
usr5_acct_results_obj_path = results_base + "/step4/Usr5__acct.json"
save_object_to_file(usr5_acct, usr5_acct_results_obj_path)
# 2. For user 2, get the identity
usr2_identity_query = {
    "type" : "identity",
    "property": {
        "path": ["extensions", "extension-definition--66e2492a-bbd3-4be6-88f5-cc91a017a498", "email_addresses", [0], "email_address_ref"],
        "source_value": usr2_email["id"],
        "comparator": "EQ"
    }
}
usr2_identity = invoke_get_from_company_block(usr2_identity_query, context_type, source_value=None, source_id=None)
usr2_identity_results_obj_path = results_base + "/step4/Usr2__identity.json"
save_object_to_file(usr2_identity, usr2_identity_results_obj_path)
# 3 identity
usr3_identity_query = {
    "type" : "identity",
    "property": {
        "path": ["extensions", "extension-definition--66e2492a-bbd3-4be6-88f5-cc91a017a498", "email_addresses", [0], "email_address_ref"],
        "source_value": usr3_email["id"],
        "comparator": "EQ"
    }
}
usr3_identity = invoke_get_from_company_block(usr3_identity_query, context_type, source_value=None, source_id=None)
usr3_identity_results_obj_path = results_base + "/step4/Usr3__identity.json"
save_object_to_file(usr3_identity, usr3_identity_results_obj_path)
# 4 identity
usr4_identity_query = {
    "type" : "identity",
    "property": {
        "path": ["extensions", "extension-definition--66e2492a-bbd3-4be6-88f5-cc91a017a498", "email_addresses", [0], "email_address_ref"],
        "source_value": usr4_email["id"],
        "comparator": "EQ"
    }
}
usr4_identity = invoke_get_from_company_block(usr4_identity_query, context_type, source_value=None, source_id=None)
usr4_identity_results_obj_path = results_base + "/step4/Usr4__identity.json"
save_object_to_file(usr4_identity, usr4_identity_results_obj_path)
# 5 identity
usr5_identity_query = {
    "type" : "identity",
    "property": {
        "path": ["extensions", "extension-definition--66e2492a-bbd3-4be6-88f5-cc91a017a498", "email_addresses", [0], "email_address_ref"],
        "source_value": usr5_email["id"],
        "comparator": "EQ"
    }
}
usr5_identity = invoke_get_from_company_block(usr5_identity_query, context_type, source_value=None, source_id=None)
usr5_identity_results_obj_path = results_base + "/step4/Usr5__identity.json"
save_object_to_file(usr5_identity, usr5_identity_results_obj_path)
# Get Exchange Identity Object
context_type = {
    "context_type": "systems"
}
exchange_identity_query = {
    "type" : "identity",
    "property": {
        "path": ["name"],
        "source_value": "Microsoft Exchange",
        "comparator": "EQ"
    }
}
exchange_identity = invoke_get_from_company_block(exchange_identity_query, context_type, source_value=None, source_id=None)
exchange_identity_results_obj_path = results_base + "/step4/Systems_Exchange__identity.json"
save_object_to_file(exchange_identity, exchange_identity_results_obj_path)
# Get Indicator Object
context_type = {
    "context_type": "other"
}
indicator_query = {
    "type" : "indicator",
    "property": {
        "path": ["name"],
        "source_value": "Potential Phishing Email",
        "comparator": "EQ"
    }
}
indicator = invoke_get_from_incident_block(indicator_query, context_type, source_value=None, source_id=None)
indicator_results_obj_path = results_base + "/step4/indicator.json"
save_object_to_file(indicator, indicator_results_obj_path)
#
# Save everything but the existing indicator down itno unattached
# context_type = {
#     "context_type": "unattached"
# }
context_records = [
    "Usr1__email",
    "Usr1__ident",
    "Usr1__usr_acct",
    "Usr2__email",
    "Usr2__ident",
    "Usr2__usr_acct",
    "Usr3__email",
    "Usr3__ident",
    "Usr3__usr_acct",
    "Usr4__email",
    "Usr4__ident",
    "Usr4__usr_acct",
    "Usr5__email",
    "Usr5__ident",
    "Usr5__usr_acct",
    "Systems_Laptop1__ident",
    "Systems_Laptop2__ident",
    "Systems_Laptop3__ident",
    "Systems_Laptop4__ident",
    "Systems_Laptop5__ident",
    "Systems_Exchange__ident"
]
for rec in context_records:
    results_path = results_base + "step0/" + rec + ".json"
    save_path = results_base + "step4/" + rec + "_context.json"
    result = invoke_save_unattached_context_block(results_path, save_path)
    print(f"{rec} result->{result}")

results_path = results_base + "step1/indicator_alert.json"
save_path = results_base + "step4/indicator_alert_context.json"
result = invoke_save_unattached_context_block(results_path, save_path)
print(f"{rec} result->{result}")

company query->{'type': 'user-account', 'property': {'path': ['id'], 'source_value': 'user-account--5c0dcb9b-5784-5aaf-b393-d990a4d68dd3', 'comparator': 'EQ'}}
company query->{'type': 'user-account', 'property': {'path': ['id'], 'source_value': 'user-account--113460bd-67f7-5611-bbbb-158d5c45255e', 'comparator': 'EQ'}}
company query->{'type': 'user-account', 'property': {'path': ['id'], 'source_value': 'user-account--2b87df75-b95c-5808-bcc7-f6d23471b9b4', 'comparator': 'EQ'}}
company query->{'type': 'user-account', 'property': {'path': ['id'], 'source_value': 'user-account--9a5926a5-b2e9-56c0-82fd-3c858bf14832', 'comparator': 'EQ'}}
company query->{'type': 'identity', 'property': {'path': ['extensions', 'extension-definition--66e2492a-bbd3-4be6-88f5-cc91a017a498', 'email_addresses', [0], 'email_address_ref'], 'source_value': 'email-addr--b0708db8-71e9-50f9-950c-610fccd2c30b', 'comparator': 'EQ'}}
company query->{'type': 'identity', 'property': {'path': ['extensions', 'extension-definiti

## B Make the SRO's to Connect all Who Receieved the Email

In [6]:

# B.1 Setup SRO objectsrelationship_type = "related-to"
relationship_type = "related-to"
sro_data_path = "SRO/Relationship/sro_related_to.json"
sro2_results_path = "step4/SRO2_related_to.json"
sro2_context_results_path = "step4/SRO2_related_to_context.json"
sro2 = invoke_sro_block(sro_data_path, sro2_results_path, usr2_email, usr1_email, relationship_type)
result = invoke_save_unattached_context_block(results_base + sro2_results_path, results_base + sro2_context_results_path)
print(f"SRO2 result->{result}")
sro3_results_path = "step4/SRO3_related_to.json"
sro3_context_results_path = "step4/SRO3_related_to_context.json"
sro3 = invoke_sro_block(sro_data_path, sro3_results_path, usr3_email, usr1_email, relationship_type)
result = invoke_save_unattached_context_block(results_base + sro3_results_path, results_base + sro3_context_results_path)
print(f"SRO3 result->{result}")
sro4_results_path = "step4/SRO4_related_to.json"
sro4_context_results_path = "step4/SRO4_related_to_context.json"
sro4 = invoke_sro_block(sro_data_path, sro4_results_path, usr4_email, usr1_email, relationship_type)
result = invoke_save_unattached_context_block(results_base + sro4_results_path, results_base + sro4_context_results_path)
print(f"SRO4 result->{result}")
sro5_results_path = "step4/SRO5_related_to.json"
sro5_context_results_path = "step4/SRO5_related_to_context.json"
sro5 = invoke_sro_block(sro_data_path, sro5_results_path, usr5_email, usr1_email, relationship_type)
result = invoke_save_unattached_context_block(results_base + sro5_results_path, results_base + sro5_context_results_path)
print(f"SRO5 result->{result}")

input file->{'relationship_form': {'base_required': {'type': 'relationship', 'spec_version': '2.1', 'id': '', 'created': '', 'modified': ''}, 'base_optional': {'created_by_ref': '', 'revoked': None, 'labels': [], 'lang': '', 'external_references': [], 'object_marking_refs': [], 'granular_markings': [], 'defanged': None}, 'object': {'relationship_type': 'related-to', 'source_ref': '', 'target_ref': '', 'start_time': None, 'stop_time': None}, 'extensions': {}, 'sub': {}}, 'source': {'type': 'email-addr', 'spec_version': '2.1', 'id': 'email-addr--b0708db8-71e9-50f9-950c-610fccd2c30b', 'value': 'sbilly@example.com', 'display_name': 'Strange Billy', 'belongs_to_ref': 'user-account--5c0dcb9b-5784-5aaf-b393-d990a4d68dd3'}, 'target': {'type': 'email-addr', 'spec_version': '2.1', 'id': 'email-addr--4722424c-7012-56b0-84d5-01d076fc547b', 'value': 'naive@example.com', 'display_name': 'Naive Smith', 'belongs_to_ref': 'user-account--597ad4d4-35ba-585d-8f6d-134a75032f9b'}, 'relationship_type': 'rela

## C. Collect the Elements of the Observation into an Observed-Data Object

Tbis then represents the potential phishing email as a group of elements

In [7]:
# 1. Setup observed-data
obs_refs3 = [usr2_acct, usr2_email, usr2_identity, usr3_email, usr3_acct, usr3_identity, usr4_email, usr4_acct, usr4_identity,usr5_email, usr5_acct, usr5_identity, sro2, sro3, sro4, sro5]
# 2. Setup path to form and results
obs_path ="SDO/ObservedData/observation-context.json"
results_path ="step3/observation-context.json"
# 3. Invoke the Make Observed Data Block
obs_3 = invoke_make_observed_data_block(obs_path, results_path, observation=obs_refs3)
# 4. Add the record to the in-session bundles and lists
context_type = {
    "context_type": "unattached"
}
obs_results_context_path = results_base + "step3/observation-context_context.json"
result = invoke_save_unattached_context_block(results_base + results_path, obs_results_context_path)
print(f" result->{result}")

{
    "type": "observed-data",
    "spec_version": "2.1",
    "id": "observed-data--ecce11ce-087e-4bb8-b6e7-8c34ad7fc9ed",
    "created": "2025-12-10T01:16:23.276Z",
    "modified": "2025-12-10T01:16:23.276Z",
    "first_observed": "2020-10-19T01:01:01Z",
    "last_observed": "2020-10-19T01:01:01Z",
    "number_observed": 1,
    "object_refs": [
        "user-account--5c0dcb9b-5784-5aaf-b393-d990a4d68dd3",
        "email-addr--b0708db8-71e9-50f9-950c-610fccd2c30b",
        "identity--a7e10775-126b-4122-825a-dda798829097",
        "email-addr--72fa6570-cce5-5be5-916d-452de0e1adb5",
        "user-account--113460bd-67f7-5611-bbbb-158d5c45255e",
        "identity--22c03ae2-f432-4fae-b6b4-645e836213e8",
        "email-addr--b8048a91-def8-5ef7-8cd3-a7b3db9278e5",
        "user-account--2b87df75-b95c-5808-bcc7-f6d23471b9b4",
        "identity--6da70496-b1fb-43c4-a39a-7894a02b218c",
        "email-addr--30d9a416-203b-55c8-b796-8eb65ab5275e",
        "user-account--9a5926a5-b2e9-56c0-82fd-3c858

## D. Create the Contextual Sighting

Tbis then represents the Anecdote on Impact from the reporting user

### D.1 Create the Sighting
Connecting the:
- observed-data object, containing the extra identitiies, email addresses and user accounts that got the email
- the exchange server as the location
- the original indicator and pattern

In [8]:

#
# D. Setup the Sighting Object
sighting_data_path ="SRO/Sighting/sighting_context.json"
results_path ="step3/sighting_context.json"
# 2. Setup the SDO sighted, the Observed-Data that was observed with generated objects,
#                        then the identity object from the context storage (note the slight difference in indexing
sighted = indicator
observation_list = [obs_3]
where_list = [exchange_identity]
# 2. Invoke the Make Observed Data Block
sight3 = invoke_sighting_block(sighting_data_path, results_path, observed=observation_list, sighted=sighted, where=where_list)
# 3. Add the record to the in-session bundles and lists
context_type = {
    "context_type": "unattached"
}
sighting_results_context_path = results_base + "step1/sighting_context_context.json"
result = invoke_save_unattached_context_block(results_base + results_path, sighting_results_context_path)
print(f" result->{result}")

<class 'dict'>
['sighting_form', 'observed_data_refs', 'where_sighted_refs', 'sighting_of_ref']
{
    "type": "sighting",
    "spec_version": "2.1",
    "id": "sighting--91031c33-a262-411d-8146-8c93a33e58f6",
    "created": "2025-12-10T01:16:23.414Z",
    "modified": "2025-12-10T01:16:23.414Z",
    "count": 1,
    "sighting_of_ref": "indicator--51b7f178-7a2d-437f-86a2-d926a77da3a4",
    "observed_data_refs": [
        "observed-data--ecce11ce-087e-4bb8-b6e7-8c34ad7fc9ed"
    ],
    "where_sighted_refs": [
        "identity--dba8a25e-a184-42a1-bce8-03e47fb06a56"
    ],
    "extensions": {
        "extension-definition--0d76d6d9-16ca-43fd-bd41-4f800ba8fc43": {
            "extension_type": "property-extension"
        },
        "sighting-context": {
            "name": "user-report",
            "description": "query from: evil@northkorea.com, subject:we are coming for you",
            "value": "sbilly@example.com, wwhilly@example.com, strange@mycompany.com, dumbo@mycompany.com"
      

### D.2 Promote The Sighting and its Components from Unattached to Other

Everything that uis connected in the Sighting should now be promoted, so move Unattached to the Other list

In [9]:
# total_list = [usr2_acct, usr2_email, usr2_identity, usr3_email, usr3_acct, usr3_identity,              usr4_email, usr4_acct, usr4_identity,usr5_email, usr5_acct, usr5_identity,
#             sro2, sro3, sro4, sro5]
# total_list.append(obs_3)
# total_list.append(indicator)
# total_list.append(sight3)
# total_list.append(exchange_identity)

# obs_context_move_path = results_base + "step3/context/context_move.json"
# obs_context_move_results = results_base + "step4/context/context_move_results.json"
# result = invoke_move_unattached_to_other_block(obs_context_move_path, obs_context_move_results, total_list)

## E. Create the Impact Objects

Tbis Impact represents the effect reported by the user in the Anecdote

In [10]:
# # 1. Setup path to form and results
# impact_path ="SDO/Impact/context_impact.json"
# results_path ="step4/impact_context.json"
# # 2. Setup the number of assets impacted
# numbers = {"computers-mobile": 5}
# impacted_refs =
# # 2. Invoke the Make Observed Data Block
# impact_1 = invoke_make_impact_block(impact_path, results_path, impacted_entity_counts=numbers, impacted_refs=impacted_refs, superseded_by_ref=None)
# # 3. Add the record to the in-session bundles and lists
# context_type = {
#     "context_type": "impact"
# }
# impact_results_obj_path = results_base + results_path
# impact_results_context_path = results_base + "/step2/impact_anecdote_context.json"
# result = invoke_save_incident_context_block(impact_results_obj_path, impact_results_context_path, context_type)
# print(f" result->{result}")

## F. Create the Next Task Object

Next step is to investigate the Exchange server to see who else got the email

### F.1 Create the Task Object

In [11]:
# # 1. Setup path to form and results
# task_data_path ="SDO/Task/task_anecdote.json"
# results_path ="step2/task_anecdote.json"
# # 2. Invoke the Make Observed Data Block
# task_2 = invoke_make_task_block(task_data_path, results_path, changed_objects=None)
# # 3. Add the record to the in-session bundles and lists
# bundle_list = bundle_list + task_2
# task_objs.append(task_2[0])
# #
# # Step 3.A.2 New Task to check te Exclusion Lists in Step 4
# #
# task3 = Task(
#     task_types=["investigation"], outcome="pending", name="Check Exclusion Lists",
#     description="Check OS-Threat Exclusion List to see if email address is a known phisher",
#     owner=me.id, extensions={task_ext_id:task_ext}
# )
# tseq1_3 = Sequence(
#     step_type="single_step", sequenced_object=task3.id,
#     sequence_type="task", extensions=seq_ext_dict
# )

### F.2 Create the Sequence Object

In [12]:
# # 1. Setup path to form and results
# sequence_data_path ="SDO/Sequence/sequence_alert.json"
# results_path ="step1/sequence_task_single.json"
# # 2. Setup the Sequence Object for the Event
# #
# step_type = "single_step"
# sequence_type = "task"
# sequenced_object = task_2[0]["id"]
# # 2. Invoke the Make Observed Data Block
# seq_T_2 = invoke_make_sequence_block(sequence_data_path, results_path, step_type=step_type, sequence_type=sequence_type, sequenced_object=sequenced_object, on_completion=None, on_success=None, on_failure=None, next_steps=None)
# # 3. Add the record to the in-session bundles and lists
# bundle_list = bundle_list + seq_T_2
# sequence_objs.append(seq_T_2[0])

### F.3 Connect the Sequence to the Previous one

In [13]:
# # Find the Previous Sequence and link it to the above Sequence
# last_task = task_objs[-1]
# for rec in sequence_objs:
#     if rec["sequence_type"] == "task" and rec["sequenced_object"] == last_task["id"]:
#         rec["on_completion"] = seq_T_2[0]["id"]

## G. Finally, Append all of the objects to the Incident Object

Assign all of the objets to the Incident Object

In [14]:
# 1. Setup path to form and results
# inc_path ="SDO/Incident/phishing_incident.json"
# results_path ="step1/incident_alert.json"
# # 2. Setup the Sequence Object for the Event
# #
# sequence_start_refs = [x["id"] for x in sequence_start_objs]
# sequence_refs = [x["id"] for x in sequence_objs]
# task_refs = [x["id"] for x in task_objs]
# event_refs = [x["id"] for x in event_objs]
# impact_refs = [x["id"] for x in impact_objs]
# other_object_refs = [x["id"] for x in other_object_objs]
# # 3. Update the actual Incident Object
# incident_obj["sequence_start_refs"] = sequence_start_refs
# incident_obj["sequence_refs"] = sequence_refs
# incident_obj["task_refs"] = task_refs
# incident_obj["event_refs"] = event_refs
# incident_obj["impact_refs"] = impact_refs
# incident_obj["other_object_refs"] = other_object_refs

## H. Write out the Contex Memory for the Incident

Export out the Context Memory for the Incident

In [15]:
# Save the Tpe Refinery Context Memory File
# invoke_context_save_block(Type_Refinery_Context)